# Semantic Search

**Retrieve by meaning, not by keyword.** Embed every document into a vector once, embed the query the same way, and return the nearest vectors by cosine similarity — so *"my sedan won't start"* finds *"the automobile would not start"* even though they share no words.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What:** Semantic search is the end-to-end *retrieval task* built on dense embeddings. Pipeline: **embed the corpus → store the vectors → embed the query → nearest-neighbour search → return top-k**. The score is usually cosine similarity (or inner product on normalized vectors).

**The problem it solves:** Classic keyword search (TF-IDF, BM25) matches on *surface tokens*. It suffers the **vocabulary mismatch** problem: the query and the relevant document describe the same thing with different words — "car"/"automobile", "lift"/"elevator", "heart attack"/"myocardial infarction". Lexical search returns nothing useful because the strings don't overlap. Semantic search compares *meaning* (geometry), so synonyms, paraphrases, and related concepts land close together.

**When to reach for it:**
- Natural-language queries over prose, FAQs, support tickets, code, transcripts.
- The **retrieval** half of RAG — fetch the passages an LLM will read (see [[rag-retrieval-augmented-generation]]).
- Recommendation / "more like this", deduplication, clustering, cross-lingual search.

**When *not* to:** exact-match needs (SKUs, error codes, legal citations, identifiers), tiny corpora where a substring scan is fine, or when users expect literal keyword behaviour. In practice the best systems are **hybrid** — lexical *and* semantic — because each covers the other's blind spot.

## 2. Mental Model

Think of a **library where books are shelved by meaning, not by title**. In a keyword library you can only find a book if you already know a word on its spine. In the semantic library, every book is placed at a coordinate in a "map of meaning"; you describe what you want in your own words, that description gets a coordinate too, and you simply walk to the nearest shelves.

```
   query: "trouble starting my sedan"
                |
                v  (same embedding model as the corpus)
        q = [ 0.21, -0.74, ... ]            map of meaning
                |                      vehicles .  . "automobile won't start"   <- nearest
                |                               . "sedan at the mechanic"
                +----- nearest neighbours -->  . "vehicle broke down"
                                          food  . "tomato sauce"   (far away)
                                       finance  . "stock market"   (far away)
```

Two halves do the work:
1. **Embedder** — turns text into a vector so that *distance ≈ dissimilarity*. This is where the "semantics" lives (see [[vector-embeddings]]).
2. **Index / nearest-neighbour search** — finds the closest vectors fast. Brute force for thousands of docs; an ANN index ([[faiss]], HNSW) for millions.

The query and the corpus **must** be embedded by the *same* model into the *same* space — that is the whole trick.

## 3. Key Concepts

- **Bi-encoder (dual encoder)** — the standard retriever. Documents and queries are embedded *independently* by the same model, so document vectors are precomputed once and reused. Fast and scalable; this is what "semantic search" usually means.
- **Cross-encoder** — feeds *query + document together* through a transformer to score relevance. Far more accurate but can't precompute, so it's used only to **rerank** the top-k a bi-encoder retrieves (see [[rerankers]]).
- **Cosine similarity / inner product** — the match score. Normalize vectors to unit length once, then cosine == dot product (cheaper). Score in [-1, 1]; higher = more similar.
- **Top-k / ANN** — return the *k* nearest vectors. Exact (brute force) is O(N·d) per query; **approximate nearest neighbour** (HNSW, IVF, ScaNN) trades a little recall for huge speedups at scale.
- **Vocabulary mismatch** — query and relevant doc use different words for the same idea. The core failure of lexical search and the core win of semantic search.
- **Chunking** — long documents are split into passages *before* embedding, because one vector can only summarise so much text. Chunk size/overlap materially affects retrieval quality.
- **Hybrid search** — combine lexical (BM25) and semantic scores (e.g. weighted sum, or Reciprocal Rank Fusion). Captures both exact matches and conceptual matches.
- **Embedding drift** — if you re-embed the corpus with a new model, you must re-embed *queries* with the same model. Mixing model versions silently destroys relevance.

## 4. Setup

The always-on examples use only **scikit-learn** and **numpy**, so they run on any CPU in seconds with no downloads or API keys. We build a real (if classic) semantic index with **LSA** — TF-IDF followed by truncated SVD — which projects sparse keyword vectors into a dense "meaning" space. The optional final cell shows the *modern* path with a pretrained neural embedder (`sentence-transformers`) and is gated so the notebook still runs end-to-end without it.

```bash
pip install scikit-learn numpy
# optional, for the real-model cell (downloads a ~90 MB model on first use):
pip install sentence-transformers
```

In [1]:
import numpy as np
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)

# A tiny corpus with three clear topics: vehicles, cycling, food, finance.
# Note the synonyms scattered across docs: car / automobile / vehicle / sedan.
corpus = [
    "I drive my car to the office on the highway every morning",
    "The automobile would not start because the engine was flooded",
    "My vehicle broke down on the road and the motor overheated",
    "Traffic on the freeway was terrible during the morning commute",
    "He took the sedan to the mechanic to fix the engine",
    "She rode her bicycle along the bike path through the park",
    "Cycling on two wheels is great exercise on a quiet trail",
    "The cyclist pedalled up the steep hill on her road bike",
    "Riding a bike to work is healthy outdoor exercise",
    "The chef cooked a delicious pasta dish with fresh tomatoes",
    "He baked sourdough bread and simmered a rich tomato sauce",
    "The restaurant served grilled salmon with a lemon butter sauce",
    "She seasoned the soup and tasted it before serving dinner",
    "Investors worried as the stock market fell sharply on Friday",
    "The central bank raised interest rates to fight rising inflation",
    "Quarterly earnings beat expectations and company shares rallied",
    "Traders sold bonds as prices dropped across the market",
]
print(f"{len(corpus)} documents")

numpy 2.5.0 | scikit-learn 1.9.0
17 documents


## 5. Worked Examples

### Example 1 — Lexical search (TF-IDF) and where it breaks

First the baseline: rank documents by **TF-IDF cosine similarity** — pure keyword overlap. It works when the query reuses the document's words, and returns *nothing* when it doesn't. Watch the second query: the relevant car documents score **0.000** because the query says "sedan", a word that appears in only one of them.

In [2]:
# Fit a TF-IDF model: each doc -> sparse vector of (weighted) word counts.
tfidf = TfidfVectorizer(stop_words="english")
X_lex = tfidf.fit_transform(corpus)          # (n_docs, n_terms), L2-normalized rows

def lexical_search(query, k=3):
    q = tfidf.transform([query])             # same vocabulary as the corpus
    sims = (X_lex @ q.T).toarray().ravel()   # cosine == dot on normalized rows
    order = np.argsort(-sims)[:k]
    return [(float(sims[i]), corpus[i]) for i in order]

for query in ["engine trouble on the highway", "trouble starting my sedan"]:
    print(f"QUERY: {query!r}")
    for score, doc in lexical_search(query):
        print(f"   {score:.3f}  {doc}")
    print()

QUERY: 'engine trouble on the highway'
   0.345  I drive my car to the office on the highway every morning
   0.296  The automobile would not start because the engine was flooded
   0.263  He took the sedan to the mechanic to fix the engine

QUERY: 'trouble starting my sedan'
   0.458  He took the sedan to the mechanic to fix the engine
   0.000  I drive my car to the office on the highway every morning
   0.000  The central bank raised interest rates to fight rising inflation



### Example 2 — Semantic search with LSA (dense vectors + cosine top-k)

Now the real thing. We project the sparse TF-IDF vectors into a dense **latent semantic** space with truncated SVD: documents that share *context* (even without sharing the exact words) get pulled together, so "automobile", "sedan", and "vehicle" end up near each other.

The retrieval loop is the universal semantic-search pattern — **embed corpus once, embed query, cosine, top-k** — and it's identical whether the embedder is LSA, a neural bi-encoder, or an embedding API.

In [3]:
# Embed the corpus ONCE into a dense d-dim "meaning" space, then normalize so
# cosine similarity is a plain dot product. LSA = TF-IDF -> truncated SVD.
svd = TruncatedSVD(n_components=7, random_state=0)
doc_vectors = normalize(svd.fit_transform(X_lex))   # (n_docs, 7), unit length

def semantic_search(query, k=3):
    q = normalize(svd.transform(tfidf.transform([query])))   # SAME pipeline as corpus
    sims = (doc_vectors @ q.T).ravel()
    order = np.argsort(-sims)[:k]
    return [(float(sims[i]), corpus[i]) for i in order]

# Same "sedan" query that scored 0.000 under lexical search:
query = "trouble starting my sedan"
print(f"QUERY: {query!r}\n")
print("lexical (TF-IDF):")
for score, doc in lexical_search(query):
    print(f"   {score:.3f}  {doc}")
print("\nsemantic (LSA):")
for score, doc in semantic_search(query):
    print(f"   {score:.3f}  {doc}")

QUERY: 'trouble starting my sedan'

lexical (TF-IDF):
   0.458  He took the sedan to the mechanic to fix the engine
   0.000  I drive my car to the office on the highway every morning
   0.000  The central bank raised interest rates to fight rising inflation

semantic (LSA):
   1.000  The automobile would not start because the engine was flooded
   1.000  He took the sedan to the mechanic to fix the engine
   0.000  My vehicle broke down on the road and the motor overheated


The semantic index pulls *"the automobile would not start"* and *"the sedan to the mechanic to fix the engine"* to the top — the right documents, with **zero shared keywords** with the query. That bridging is exactly what lexical search cannot do.

One honest caveat visible above: LSA learns its "meaning" purely from *this* 17-document corpus, so the latent topics are coarse and the scores are noisy. A pretrained neural embedder has read billions of sentences and generalises far better — that's the next cell.

In [4]:
# ---- The modern path: a real neural bi-encoder, gated so it is optional ----
# sentence-transformers downloads a ~90 MB model on first run, so we skip it
# cleanly when the package is missing. The retrieval loop is IDENTICAL to above.
try:
    from sentence_transformers import SentenceTransformer
    _HAVE_ST = True
except ImportError:
    _HAVE_ST = False

if _HAVE_ST:
    model = SentenceTransformer("all-MiniLM-L6-v2")          # 384-dim bi-encoder
    doc_emb = model.encode(corpus, normalize_embeddings=True)  # embed corpus once

    def neural_search(query, k=3):
        q = model.encode([query], normalize_embeddings=True)
        sims = (doc_emb @ q.T).ravel()
        order = np.argsort(-sims)[:k]
        return [(float(sims[i]), corpus[i]) for i in order]

    query = "trouble starting my sedan"
    print(f"QUERY: {query!r}  (real neural embeddings)\n")
    for score, doc in neural_search(query):
        print(f"   {score:.3f}  {doc}")
else:
    print("sentence-transformers not installed - skipping the neural example.")
    print("Install it (`pip install sentence-transformers`) to run real embeddings.")
    print("\nThe call shape is exactly:")
    print('   model = SentenceTransformer("all-MiniLM-L6-v2")')
    print('   doc_emb = model.encode(corpus, normalize_embeddings=True)')
    print('   q = model.encode([query], normalize_embeddings=True)')
    print('   sims = doc_emb @ q.T   # cosine; take the top-k')

sentence-transformers not installed - skipping the neural example.
Install it (`pip install sentence-transformers`) to run real embeddings.

The call shape is exactly:
   model = SentenceTransformer("all-MiniLM-L6-v2")
   doc_emb = model.encode(corpus, normalize_embeddings=True)
   q = model.encode([query], normalize_embeddings=True)
   sims = doc_emb @ q.T   # cosine; take the top-k


### Example 3 — Hybrid search: fuse lexical and semantic ranks

Neither signal wins everywhere — lexical nails exact tokens (codes, names), semantic nails paraphrase. **Reciprocal Rank Fusion (RRF)** is the standard, score-scale-free way to merge two ranked lists: each document scores `Σ 1/(c + rank)` across the rankers it appears in. No tuning of incompatible score ranges required.

In [5]:
def rrf(query, k=4, c=60):
    """Fuse lexical + semantic rankings with Reciprocal Rank Fusion."""
    # rank every doc under each retriever (full lists, so every doc gets a rank)
    lex = [corpus.index(d) for _, d in lexical_search(query, k=len(corpus))]
    sem = [corpus.index(d) for _, d in semantic_search(query, k=len(corpus))]
    scores = {}
    for ranked in (lex, sem):
        for rank, doc_i in enumerate(ranked):
            scores[doc_i] = scores.get(doc_i, 0.0) + 1.0 / (c + rank)
    top = sorted(scores, key=scores.get, reverse=True)[:k]
    return [(scores[i], corpus[i]) for i in top]

query = "engine trouble with my automobile on the freeway"
print(f"QUERY: {query!r}  (hybrid RRF)\n")
for score, doc in rrf(query):
    print(f"   {score:.4f}  {doc}")

QUERY: 'engine trouble with my automobile on the freeway'  (hybrid RRF)

   0.0333  The automobile would not start because the engine was flooded
   0.0325  Traffic on the freeway was terrible during the morning commute
   0.0325  He took the sedan to the mechanic to fix the engine
   0.0317  I drive my car to the office on the highway every morning


## 6. Gotchas & Pitfalls

- **Query/corpus model mismatch.** Embedding the query with a different model (or version) than the corpus silently wrecks relevance — the vectors live in different spaces. Re-embed *both* together whenever you change models.
- **Out-of-vocabulary kills LSA / TF-IDF.** Bag-of-words methods can't embed a word they never saw in the corpus — its vector is zero. (Try a query of pure novel synonyms against the LSA index: 0.000 everywhere.) Neural embedders use subword tokens and context, so they generalise to unseen words. This is the main reason to prefer pretrained models in production.
- **Forgetting to normalize.** If you intend cosine but feed un-normalized vectors to an inner-product index, longer vectors dominate. Normalize once at index time *and* query time.
- **Bad chunking.** Embedding whole long documents blurs many topics into one vector; chunks too small lose context. Tune chunk size/overlap — it often matters more than the model choice.
- **Semantic-only blind spots.** Pure embedding search is weak on exact identifiers, rare proper nouns, and negation ("not waterproof" embeds near "waterproof"). Add lexical/hybrid and, for precision, a reranker.
- **Recall vs latency at scale.** ANN indexes (HNSW/IVF) are approximate — aggressive settings drop the true nearest neighbour. Measure recall@k, don't assume it's exact.
- **Stale index.** Adding documents to the corpus without embedding and inserting them means they're invisible to search. Keep the index in sync with the source of truth.
- **Cosine is not calibrated.** A 0.8 from one model is not "80% relevant" and isn't comparable across models. Use it only to *rank*; pick thresholds empirically per model.

## 7. When to Use vs Alternatives

| Approach | Best at | Weak at | Use when |
|---|---|---|---|
| **Lexical (BM25 / TF-IDF)** | Exact terms, codes, names; zero training; cheap, interpretable | Synonyms & paraphrase (vocabulary mismatch) | Identifiers, short queries, keyword-literal expectations |
| **Semantic (bi-encoder)** | Meaning, paraphrase, natural-language & cross-lingual queries | Exact tokens, rare proper nouns, negation; needs a model | NL search over prose, RAG retrieval, "more like this" |
| **Hybrid (BM25 + dense, RRF)** | Robust across both query types — usually the best overall recall | More moving parts to build & tune | Production search where queries vary; default for serious systems |
| **Cross-encoder reranker** | Highest precision on the top-k | Too slow to score the whole corpus | Rerank the top 50–200 candidates after retrieval ([[rerankers]]) |

**Rules of thumb:**
- The strong default is **retrieve hybrid → rerank with a cross-encoder**. Bi-encoder (or BM25+dense) for recall, cross-encoder for precision.
- Don't replace keyword search — *augment* it. Hybrid beats either alone on most benchmarks.
- For millions of vectors, reach for an ANN index ([[faiss]], HNSW) or a managed vector DB; brute-force numpy is fine into the tens of thousands.
- Semantic search is the *retrieval* component of [[rag-retrieval-augmented-generation]] — the same embed-and-rank machinery feeds the LLM's context.

## 8. Resources

- **Sentence-Transformers — Semantic Search docs** — the canonical bi-encoder/cross-encoder retrieval guide with runnable recipes: https://www.sbert.net/examples/applications/semantic-search/README.html
- **Pretrained models & MTEB leaderboard** — pick an embedding model by task, size, and benchmark score: https://huggingface.co/spaces/mteb/leaderboard
- **BM25 + dense hybrid / RRF** — Elastic's explainer on reciprocal rank fusion and why hybrid wins: https://www.elastic.co/search-labs/blog/articles/hybrid-search-rrf
- **FAISS wiki** — building and tuning approximate-nearest-neighbour indexes for scale: https://github.com/facebookresearch/faiss/wiki
- **Pinecone — Semantic search learning series** — concepts, chunking, and end-to-end pipelines: https://www.pinecone.io/learn/what-is-semantic-search/
- **Related notebooks:** [[vector-embeddings]] (the representation) · [[faiss]] (the index) · [[rerankers]] (precision stage) · [[rag-retrieval-augmented-generation]] (the consumer).